# ERA5 Weather Data Analysis

This notebook explores ERA5 reanalysis weather data using Python. The analysis focuses on grid structure, geographic selection around Fulda, temperature fields, interpolation effects, and multivariate visualisation of atmospheric variables.

## Project goals

- Load and inspect NetCDF weather datasets.
- Identify the ERA5 grid point closest to Fulda, Germany.
- Analyse temperature at selected pressure levels and time steps.
- Visualise global and regional weather fields.
- Compare interpolation methods and grid resolutions.
- Explore relationships between atmospheric variables such as temperature, humidity, and wind components.

## Data files expected

Place the following files inside a local `data/` folder before running this notebook:

```text
data/ERA5_2026-04-22_hourly.nc
data/ERA5_2026-04-22_hourly_1deg.nc
data/dv_ex4_data.csv
```

> The raw data files are not included in this repository because NetCDF weather files can be large.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Reusable project configuration
DATA_DIR = Path('data')
ERA5_025_FILE = DATA_DIR / 'ERA5_2026-04-22_hourly.nc'
ERA5_1DEG_FILE = DATA_DIR / 'ERA5_2026-04-22_hourly_1deg.nc'
DV_EX4_FILE = DATA_DIR / 'dv_ex4_data.csv'

FULDA_LAT = 50.55
FULDA_LON = 9.68
ANALYSIS_TIME = '2026-04-22T12:00:00'

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.grid'] = True


In [ ]:
required_files = [ERA5_025_FILE, ERA5_1DEG_FILE, DV_EX4_FILE]
missing_files = [str(path) for path in required_files if not path.exists()]

if missing_files:
    print('Missing data files:')
    for file in missing_files:
        print(f' - {file}')
    print('Please add these files before running the full notebook.')
else:
    print('All required data files are available.')


## 1. Load and inspect the ERA5 dataset



In [ ]:
ds = xr.open_dataset(ERA5_025_FILE)
ds


In [ ]:
print(ds.dims)
print(ds.coords)

In [ ]:
!ncdump -h {ERA5_1DEG_FILE}

## Finding grid characteristics using ncdump and xarray

Both tools expose the **dimensions, coordinates, and attributes** of a NetCDF file, which together describe the grid. They give the same underlying information in different forms.

**Using `ncdump` (command line):** running ncdump -h filename.nc prints the file *header* the dimensions, variables, their attached dimensions, and metadata — without dumping the actual data values. The `-h` (header-only) flag makes this practical for large files.

**Using `xarray` (Python):** loading the file with xr.open_dataset(filename) and inspecting the resulting object such as ds.dims, ds.coords, and ds.<var>.attrs, shows the same dimension sizes, coordinate arrays, and attributes interactively.

### Grid characteristics observed

The dataset has four dimensions, each attached to the data variables (t, q, u, v, w, …):

| Dimension | Size | Range | Spacing |
|-----------|------|-------|---------|
| `latitude` | 721 | 90°N → 90°S | 0.25° |
| `longitude` | 1440 | 0° → 359.75°E | 0.25° |
| `pressure_level` | 37 | 1000 → 1 hPa | irregular (denser near surface) |
| `valid_time` | 24 | 00:00 → 23:00 UTC, 22 Apr 2026 | 1 hour |

### Interpretation

- **Horizontal grid:** 721 × 1440 points at **0.25° spacing** (≈28 km) → high-resolution global coverage.
- **Vertical grid:** 37 pressure levels from the surface (1000 hPa) to the top of the atmosphere (1 hPa), spaced more densely near the surface.
- **Temporal grid:** 24 hourly steps covering one full day.

Together: a **high-resolution (0.25°), hourly, global, multi-level** atmospheric dataset.

## 2. Geographic domain, grid spacing, and coordinate system

**Geographic domain:** The data covers the **entire globe**. Latitude spans 90°N to 90°S (pole to pole) and longitude spans 0° to 359.75°E (full wrap-around), so there are no regional gaps, every point on Earth's surface is represented.

**Grid spacing:** The horizontal grid has a uniform spacing of **0.25° in both latitude and longitude** (721 latitude points × 1440 longitude points). At the equator this corresponds to roughly **28 km** between grid points, giving a high-resolution view of the atmosphere.

**Coordinate system:** The data is defined on a **regular latitude–longitude (geographic) grid**, using geographic coordinates (degrees of latitude and longitude) referenced to the Earth's surface. This is an *equirectangular* arrangement, equal angular spacing in degrees.

## 3. Fulda grid-point selection

In [ ]:
lat_idx = int(np.abs(ds.latitude.values - FULDA_LAT).argmin())
lon_idx = int(np.abs(ds.longitude.values - FULDA_LON).argmin())

nearest_lat = float(ds.latitude.values[lat_idx])
nearest_lon = float(ds.longitude.values[lon_idx])

print(f'Nearest latitude: {nearest_lat} at index {lat_idx}')
print(f'Nearest longitude: {nearest_lon} at index {lon_idx}')


## 4. Temperature at Fulda at 850 hPa and 700 hPa


In [ ]:

point = ds.sel(latitude=FULDA_LAT, longitude=FULDA_LON, method='nearest')

t_850 = point['t'].sel(valid_time=ANALYSIS_TIME, pressure_level=850.0).values - 273.15
t_700 = point['t'].sel(valid_time=ANALYSIS_TIME, pressure_level=700.0).values - 273.15

print(f"Fulda grid point: {point.latitude.values}°N, {point.longitude.values}°E")
print(f"Temperature at 850 hPa: {t_850:.2f} °C")
print(f"Temperature at 700 hPa: {t_700:.2f} °C")

Local time in Fulda at 12:00 UTC:

Fulda is in Germany, which on 22 April is on Central European Summer Time (CEST = UTC+2) , daylight saving is in effect from late March to late October

So 12:00 UTC = 14:00 (2:00 PM) local time in Fulda.

## 5. Temporal evolution at the Fulda grid point, 1000 hPa

Temperature at 1000 hPa over 22 April 2026 follows the natural **diurnal (daily) cycle** 

**Temporal spacing:** the data is available at **1-hour** intervals (24 steps).

In [ ]:
fulda_point = ds.sel(latitude=FULDA_LAT, longitude=FULDA_LON, method='nearest')
t_1000 = fulda_point['t'].sel(pressure_level=1000.0) - 273.15

plt.figure(figsize=(10, 4))
t_1000.plot(marker='o')
plt.title('Temperature evolution at Fulda grid point, 1000 hPa')
plt.ylabel('Temperature (°C)')
plt.xlabel('Time')
plt.show()


## 6. Distance between neighbouring grid points




In [ ]:
import numpy as np

# Earth's radius in km
R = 6371.0

# Fulda grid point
lat1 = np.radians(51.0)
lon1 = np.radians(10.0)

# Neighbour to the east (same lat, +1° lon)
lat2 = np.radians(51.0)
lon2 = np.radians(10.25)

# Haversine formula
dlat = lat2 - lat1
dlon = lon2 - lon1
a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
dist_east = 2 * R * np.arcsin(np.sqrt(a))

print(f"Distance to eastern neighbour: {dist_east:.2f} km")


# Distance between Equator and North Pole at same longitude (10°E)
lat1 = np.radians(0)    # Equator
lat2 = np.radians(90)   # North Pole
lon1 = np.radians(10.0) # same longitude
lon2 = np.radians(10.0) # same longitude

dlat = lat2 - lat1
dlon = lon2 - lon1  # = 0 (same longitude)

a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
d = 2 * R * np.arcsin(np.sqrt(a))
print(f"Distance between Equator and North Pole: {d:.2f} km")

The distance between two grid points is calculated using the **Haversine formula**, which gives the great-circle distance between two points on a sphere from their latitude/longitude.

**Results:**
- Fulda and nearest neighbour to the east: **17.49 km**
- Equator to North Pole: **10 007.54 km**


In [ ]:
ds_low = xr.open_dataset(ERA5_025_FILE)

## 7. Global temperature map at 850 hPa

In [ ]:
import matplotlib.pyplot as plt


t_850_global = ds_low['t'].sel(valid_time=ANALYSIS_TIME, 
                                 pressure_level=850.0) - 273.15


plt.figure(figsize=(14, 6))
plt.imshow(t_850_global.values, 
           interpolation='nearest',
           extent=[0, 359, -90, 90],
           origin='upper',
           cmap='RdBu_r',
           aspect='auto')
plt.colorbar(label='Temperature (°C)')
plt.title('ERA-5 Temperature at 850 hPa - 22 April 2026 12:00 UTC\n(Nearest-neighbour interpolation)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

## 8. Regional temperature map around Fulda

In [ ]:

t_850_fulda = ds['t'].sel(
    valid_time=ANALYSIS_TIME,
    pressure_level=850.0,
    latitude=slice(53, 49),   
    longitude=slice(8, 12)   
) - 273.15

plt.figure(figsize=(6, 6))
plt.imshow(t_850_fulda.values,
           interpolation='nearest',
           extent=[8, 12, 49, 53],
           origin='upper',
           cmap='RdBu_r',
           aspect='auto')
plt.colorbar(label='Temperature (°C)')
plt.title('5x5 Grid around Fulda - 850 hPa\n(Nearest-neighbour)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
lat_vals = t_850_fulda.latitude.values
lon_vals = t_850_fulda.longitude.values

lat_indices = [int(np.where(ds.latitude.values == v)[0][0]) for v in lat_vals]
lon_indices = [int(np.where(ds.longitude.values == v)[0][0]) for v in lon_vals]

print("Latitude values: ", lat_vals,  " indices:", lat_indices)
print("Longitude values:", lon_vals, " indices:", lon_indices)

## 9. Comparing interpolation methods

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

interpolations = ['nearest', 'bilinear', 'bicubic']

for ax, interp in zip(axes, interpolations):
    ax.imshow(t_850_fulda.values,
              interpolation=interp,
              interpolation_stage='data',
              extent=[8, 12, 49, 53],
              origin='upper',
              cmap='RdBu_r',
              aspect='auto')
    ax.set_title(f'{interp.capitalize()}')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle('5x5 Grid around Fulda - 850 hPa\nInterpolation Comparison')
plt.tight_layout()
plt.show()

## 10. Interpolation effects: global view vs regional view

In [ ]:
all_interps = ['nearest', 'bilinear', 'bicubic', 'lanczos', 'sinc', 'spline16', 'spline36', 'hanning', 'hamming']

fig, axes = plt.subplots(2, len(all_interps), figsize=(20, 6))

for col, interp in enumerate(all_interps):
    # Fulda 5x5
    axes[1, col].imshow(t_850_fulda.values,
                        interpolation=interp,
                        interpolation_stage='data',
                        origin='upper',
                        cmap='RdBu_r',
                        aspect='auto')
    axes[1, col].axis('off')
    
    # Global map
    axes[0, col].imshow(t_850_global.values,
                        interpolation=interp,
                        interpolation_stage='data',
                        origin='upper',
                        cmap='RdBu_r',
                        aspect='auto')
    axes[0, col].set_title(interp, fontsize=8)
    axes[0, col].axis('off')

    

axes[0, 0].set_ylabel('Global', fontsize=9)
axes[1, 0].set_ylabel('Fulda 5x5', fontsize=9)
plt.suptitle('All Interpolation Methods - 850 hPa Temperature')
plt.tight_layout()
plt.show()

**How large are the differences, and why?**

The interpolation methods differ **dramatically for the 5×5 Fulda region** but are **almost imperceptible for the entire global grid**.

The reason is the ratio of grid cells to screen pixels. In the 5×5 region, only 25 data values are stretched across thousands of pixels, so each method has to "invent" a lot of pixels between very few data points, nearest-neighbour shows hard blocky squares, while bilinear/bicubic smooth heavily across the large gaps, making the methods look very different.

For the global grid (721×1440 points), there are roughly as many data cells as screen pixels, so almost no interpolation is needed to fill the display, every method maps nearly one data point per pixel, leaving little room to differ. The choice of interpolation matters most when **data is sparse relative to the display size**, and least when the data already resolves the figure.

## 11. Comparing grid resolutions



In [ ]:
import xarray as xr

ds_1 = xr.open_dataset(ERA5_1DEG_FILE)
ds_1

In [ ]:
# 5x5 region around Fulda (51N, 10E)
# 2 grid points in each direction
t_850_fulda_1 = ds_1['t'].sel(
    valid_time=ANALYSIS_TIME,
    pressure_level = 850.0, 
    latitude=slice(53, 49),   # 2 above, 2 below 51N
    longitude=slice(8, 12)    # 2 left, 2 right of 10E
) - 273.15

plt.figure(figsize=(6, 6))
plt.imshow(t_850_fulda_1.values,
           interpolation='nearest',
           extent=[8, 12, 49, 53],
           origin='upper',
           cmap='RdBu_r',
           aspect='auto')
plt.colorbar(label='Temperature (°C)')
plt.title('5x5 Grid around Fulda - 850 hPa\n(Nearest-neighbour)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

interpolations = ['nearest', 'bilinear', 'bicubic']

for ax, interp in zip(axes, interpolations):
    ax.imshow(t_850_fulda_1.values,
              interpolation=interp,
              interpolation_stage='data',
              extent=[8, 12, 49, 53],
              origin='upper',
              cmap='RdBu_r',
              aspect='auto')
    ax.set_title(f'{interp.capitalize()}')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle('5x5 Grid around Fulda - 850 hPa\nInterpolation Comparison')
plt.tight_layout()
plt.show()

In [ ]:
all_interps = ['nearest', 'bilinear', 'bicubic', 'lanczos', 'sinc', 'spline16', 'spline36', 'hanning', 'hamming']

fig, axes = plt.subplots(2, len(all_interps), figsize=(20, 6))

for col, interp in enumerate(all_interps):
    # Global map
    axes[0, col].imshow(t_850_global.values,
                        interpolation=interp,
                        interpolation_stage='data',
                        origin='upper',
                        cmap='RdBu_r',
                        aspect='auto')
    axes[0, col].set_title(interp, fontsize=8)
    axes[0, col].axis('off')

    # Fulda 5x5
    axes[1, col].imshow(t_850_fulda_1.values,
                        interpolation=interp,
                        interpolation_stage='data',
                        origin='upper',
                        cmap='RdBu_r',
                        aspect='auto')
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Global', fontsize=9)
axes[1, 0].set_ylabel('Fulda 5x5', fontsize=9)
plt.suptitle('All Interpolation Methods - 850 hPa Temperature')
plt.tight_layout()
plt.show()

In [ ]:

t_850_global_hires = ds['t'].sel(valid_time=ANALYSIS_TIME, 
                                        pressure_level=850.0) - 273.15


t_850_fulda_hires = ds['t'].sel(
    valid_time=ANALYSIS_TIME,
    pressure_level=850.0,
    latitude=slice(53, 49),
    longitude=slice(8, 12)
) - 273.15

print("Lower grid spacing Fulda shape:", t_850_fulda_hires.shape)

In [ ]:
# Load high-res dataset
t_850_global_hires = ds_1['t'].sel(valid_time=ANALYSIS_TIME, 
                                        pressure_level=850.0) - 273.15

# Fulda 5x5 - now needs more grid points to cover same area (0.25° spacing = 20 points for 5°)
t_850_fulda_hires = ds_1['t'].sel(
    valid_time=ANALYSIS_TIME,
    pressure_level=850.0,
    latitude=slice(53, 49),
    longitude=slice(8, 12)
) - 273.15

print("Higher grid Fulda shape:", t_850_fulda_hires.shape)

**How many grid points cover the same area?**

A 5×5 (25-point) region at **1° spacing** covers the same ground area that needs roughly **17×17 ≈ 289** points at 0.25° spacing — because each 1° cell spans ~4× the distance.

**Observation:** Coarser spacing needs far fewer points for the same area, but loses spatial detail. With the data now sparse relative to the display, the `imshow()` interpolation methods again differ noticeably.

In [ ]:
# Task 04-01

## 12. Additional visualisation exercise: loading point datasets

In [ ]:
import numpy as np

raw = np.loadtxt(
    DV_EX4_FILE,
    delimiter=",",
    dtype=str,
    skiprows=1
)

labels_col = raw[:, 0]
xy         = raw[:, 1:].astype(float)

# Check actual unique names first
print(np.unique(labels_col))

# Build dicts using actual names from data
x_arrays = {name: xy[labels_col == name, 0] for name in np.unique(labels_col)}
y_arrays = {name: xy[labels_col == name, 1] for name in np.unique(labels_col)}

# Access like:
print(x_arrays)
print(y_arrays)

## 13. Comparing descriptive statistics

In [ ]:
import matplotlib.pyplot as plt

labels = list('abcdefghijklm')

means_x = [x_arrays[name].mean() for name in labels]
means_y = [y_arrays[name].mean() for name in labels]
std_x =   [x_arrays[name].std()  for name in labels]
std_y =   [y_arrays[name].std()  for name in labels]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

axes[0,0].bar(labels, means_x)
axes[0,0].set_title('Mean of x-coordinates')

axes[0,1].bar(labels, means_y)
axes[0,1].set_title('Mean of y-coordinates')

axes[1,0].bar(labels, std_x)
axes[1,0].set_title('Std Dev of x-coordinates')

axes[1,1].bar(labels, std_y)
axes[1,1].set_title('Std Dev of y-coordinates')

plt.tight_layout()
plt.show()

### What do you notice in these plots?

By looking at the plots, the mean of all x-coordinates for different datasets, appears to be the name with `Mean x ≈ 54 for all datasets`

Also, the mean of y-coordinates also takes same values `Mean y ≈ 47 for all datasets`

The same is the case for the standard deviation for the x and y.
`Std x ≈ 17 for all datasets` and `Std y ≈ 27 for all datasets`

Therefore, this give us the impression that all the 13 datasets are identical from the statistical sense.




## 14. Scatter plots of the point datasets

Producing scatter plots for the points

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(15, 9))
axes = axes.flatten()

for i, name in enumerate(labels):
    axes[i].scatter(x_arrays[name], y_arrays[name], s=5)
    axes[i].set_title(f'Dataset {name}')

plt.tight_layout()
plt.show()

## 15. Why visualisation matters

How does visualisation helps?

From the statistical point of view, every dataset has nearly identical means and standard deviations, but visualising the dataset tells us different story. These datasets form completely different shapes:

a: dinosaur shape
b: random scatter
c: horizontal lines
d: vertical lines
e: X shape
f: star
g: random scatter
h: three clusters
i: circle
j: circle with inner ring

This points us towards an important observation that:

summary statistics can be completely blind to the actual structure of data. Two datasets can be statistically identical yet look nothing alike. So,the key takeaway for me from this dataset is "Visualizing the dataset releaves hidden patterns that cannot be viewed through the lens of statistics. *Hence visulaization plays an important role in analyzing te data*.


## 16. Multivariate atmospheric relationships in the Fulda region

In [ ]:
import xarray as xr

ds_1 = xr.open_dataset(ERA5_1DEG_FILE)


In [ ]:
print(ds_1.data_vars)

In [ ]:
sel = dict(valid_time=ANALYSIS_TIME, pressure_level=850.0,
           latitude=slice(53, 49), longitude=slice(8, 12))

t    = ds_1['t'].sel(**sel).values.flatten() - 273.15
q    = ds_1['q'].sel(**sel).values.flatten()
u    = ds_1['u'].sel(**sel).values.flatten()
v    = ds_1['v'].sel(**sel).values.flatten()
w    = ds_1['w'].sel(**sel).values.flatten()

# Scatterplot pairs
pairs = [('t', t, 'q', q), ('u', u, 'v', v), ('t', t, 'w', w), ('q', q, 'w', w)]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for i, (xlabel, x, ylabel, y) in enumerate(pairs):
    axes[i].scatter(x, y)
    axes[i].set_xlabel(xlabel)
    axes[i].set_ylabel(ylabel)

plt.tight_layout()
plt.show()


### What do i learn from the visualization?

By looking at this smaller sub region, the 5 * 5 window, we could infer a weak positive and weak negative relationship. But, the data points are too small to infer anything meaningful.


## 17. Multivariate atmospheric relationships globally

In [ ]:
sel = dict(valid_time=ANALYSIS_TIME, pressure_level=850.0)

t = ds_1['t'].sel(**sel).values.flatten() - 273.15
q = ds_1['q'].sel(**sel).values.flatten()
u = ds_1['u'].sel(**sel).values.flatten()
v = ds_1['v'].sel(**sel).values.flatten()
w = ds_1['w'].sel(**sel).values.flatten()

pairs = [('t', t, 'q', q), ('u', u, 'v', v), ('t', t, 'w', w), ('q', q, 'w', w)]

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

for i, (xlabel, x, ylabel, y) in enumerate(pairs):
    axes[i].scatter(x, y, s=1, alpha=0.3)
    axes[i].set_xlabel(xlabel)
    axes[i].set_ylabel(ylabel)

plt.tight_layout()
plt.show()

### What changes?

By considering the global dataset, we could see a clear positive trend between "t" and "q". As the temperature increases so does the specific humidity. 

And as far the "eastwind - northwind", the get a blob like structure and this makes sense because eastwind and northwind are two independent components and they dont have any relationships between them

w 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sel = dict(valid_time=ANALYSIS_TIME, pressure_level=850.0,
           latitude=slice(53, 49), longitude=slice(8, 12))

sub = ds['t'].sel(**sel)
lat2d = sub.latitude.broadcast_like(sub).values.flatten()

# Build numpy arrays
var_names = ['t', 'q', 'u', 'v', 'w', 'ciwc', 'clwc']
arrays = {
    't':    ds['t'].sel(**sel).values.flatten() - 273.15,
    'q':    ds['q'].sel(**sel).values.flatten(),
    'u':    ds['u'].sel(**sel).values.flatten(),
    'v':    ds['v'].sel(**sel).values.flatten(),
    'w':    ds['w'].sel(**sel).values.flatten(),
    'ciwc': ds['ciwc'].sel(**sel).values.flatten(),
    'clwc': ds['clwc'].sel(**sel).values.flatten(),
}

# Stack into 2D array and remove NaN rows
matrix = np.column_stack([arrays[v] for v in var_names])
lat_col = lat2d.copy()

# Remove NaN rows
valid = ~np.isnan(matrix).any(axis=1)
matrix  = matrix[valid]
lat_col = lat_col[valid]

# Normalize lat for colormap
lat_norm = (lat_col - lat_col.min()) / (lat_col.max() - lat_col.min())
colors   = plt.cm.viridis(lat_norm)

# --- Plot scatter matrix ---
n = len(var_names)
fig, axes = plt.subplots(n, n, figsize=(14, 14))

for i in range(n):
    for j in range(n):
        ax = axes[i, j]
        if i == j:
            # Diagonal: histogram
            ax.hist(matrix[:, i], bins=20, color="steelblue", edgecolor="none")
        else:
            # Off-diagonal: scatter
            ax.scatter(matrix[:, j], matrix[:, i],
                       s=15, alpha=1.0, color=colors)

        # Labels on edges only
        if i == n - 1:
            ax.set_xlabel(var_names[j], fontsize=8)
        if j == 0:
            ax.set_ylabel(var_names[i], fontsize=8)

        ax.tick_params(labelsize=6)

plt.suptitle("ERA-5 Scatterplot Matrix — Fulda region, 850 hPa, 22 Apr 2026 12 UTC",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
sub = ds['t'].sel(valid_time=ANALYSIS_TIME, pressure_level=850.0,
                  latitude=slice(53, 49), longitude=slice(8, 12))
print(sub.shape)            # should now be (5, 5)
print(sub.latitude.values)  # should show [53. 52. 51. 50. 49.]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sel = dict(valid_time=ANALYSIS_TIME,
           latitude=slice(53, 49), longitude=slice(8, 12))

sub    = ds['t'].sel(**sel)   # dims: (pressure_level, lat, lon)
pres3d = sub.pressure_level.broadcast_like(sub).values.flatten()

# Build numpy arrays
var_names = ['pressure', 't', 'q', 'u', 'v', 'w', 'ciwc', 'clwc']
arrays = {
    'pressure': pres3d,
    't':        ds['t'].sel(**sel).values.flatten() - 273.15,
    'q':        ds['q'].sel(**sel).values.flatten(),
    'u':        ds['u'].sel(**sel).values.flatten(),
    'v':        ds['v'].sel(**sel).values.flatten(),
    'w':        ds['w'].sel(**sel).values.flatten(),
    'ciwc':     ds['ciwc'].sel(**sel).values.flatten(),
    'clwc':     ds['clwc'].sel(**sel).values.flatten(),
}

# Stack into 2D array and remove NaN rows
matrix   = np.column_stack([arrays[v] for v in var_names])
pres_col = pres3d.copy()

valid    = ~np.isnan(matrix).any(axis=1)
matrix   = matrix[valid]
pres_col = pres_col[valid]

# Normalize pressure for colormap (viridis_r: high pressure = yellow)
pres_norm = (pres_col - pres_col.min()) / (pres_col.max() - pres_col.min())
colors    = plt.cm.viridis_r(pres_norm)

# --- Plot scatter matrix ---
n = len(var_names)
fig, axes = plt.subplots(n, n, figsize=(16, 16))

for i in range(n):
    for j in range(n):
        ax = axes[i, j]
        if i == j:
            ax.hist(matrix[:, i], bins=20, color="steelblue", edgecolor="none")
        else:
            ax.scatter(matrix[:, j], matrix[:, i],
                       s=18, alpha=0.6, color=colors)

        if i == n - 1:
            ax.set_xlabel(var_names[j], fontsize=8)
        if j == 0:
            ax.set_ylabel(var_names[i], fontsize=8)

        ax.tick_params(labelsize=6)

plt.suptitle("ERA-5 Scatterplot Matrix — 5×5 Fulda, ALL levels, 22 Apr 2026 12 UTC",
             fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# pick one variable pair to experiment with (e.g. t vs u)
x = df['t']
y = df['u']
pressure = df['pressure']

# --- map pressure to SIZE ---
# scale pressure to a sensible marker-size range
sizes = (pressure - pressure.min()) / (pressure.max() - pressure.min())  # 0..1
sizes = sizes * 80 + 5   # marker sizes roughly 5..85

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. different marker types
for ax, marker in zip(axes, ['o', 's', 'x']):
    ax.scatter(x, y, s=sizes, marker=marker, alpha=0.4)
    ax.set_title(f"marker = '{marker}'")
    ax.set_xlabel('t'); ax.set_ylabel('u')

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sel = dict(valid_time=ANALYSIS_TIME,
           latitude=slice(60, 35),
           longitude=slice(0, 30))

sub    = ds['t'].sel(**sel)
pres3d = sub.pressure_level.broadcast_like(sub).values.flatten()

# Build numpy arrays
var_names = ['pressure', 't', 'q', 'u', 'v', 'w', 'ciwc', 'clwc']
arrays = {
    'pressure': pres3d,
    't':        ds['t'].sel(**sel).values.flatten() - 273.15,
    'q':        ds['q'].sel(**sel).values.flatten(),
    'u':        ds['u'].sel(**sel).values.flatten(),
    'v':        ds['v'].sel(**sel).values.flatten(),
    'w':        ds['w'].sel(**sel).values.flatten(),
    'ciwc':     ds['ciwc'].sel(**sel).values.flatten(),
    'clwc':     ds['clwc'].sel(**sel).values.flatten(),
}

# Remove NaN rows (replaces pd.DataFrame.dropna())
matrix = np.column_stack([arrays[v] for v in var_names])
valid  = ~np.isnan(matrix).any(axis=1)
matrix = matrix[valid]

# Extract columns by name
idx      = {name: i for i, name in enumerate(var_names)}
pressure = matrix[:, idx['pressure']]
x        = matrix[:, idx['t']]
y        = matrix[:, idx['u']]

# ---- 1. Different COLOUR maps ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, cmap in zip(axes, ['viridis_r', 'coolwarm_r', 'plasma_r']):
    sc = ax.scatter(x, y, c=pressure, cmap=cmap, s=15, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='pressure (hPa)')
    ax.set_title(f"cmap = '{cmap}'")
    ax.set_xlabel('t (°C)')
    ax.set_ylabel('u (m/s)')
plt.suptitle("ERA-5 t vs u — Europe region, all levels, 22 Apr 2026 12 UTC")
plt.tight_layout()
plt.show()

# ---- 2. Different VALUE RANGES (vmin/vmax) ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ranges = [(0, 1000), (700, 1000), (850, 1000)]
for ax, (vmin, vmax) in zip(axes, ranges):
    sc = ax.scatter(x, y, c=pressure, cmap='viridis_r',
                    vmin=vmin, vmax=vmax, s=15, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='pressure (hPa)')
    ax.set_title(f"range {vmin}–{vmax} hPa")
    ax.set_xlabel('t (°C)')
    ax.set_ylabel('u (m/s)')
plt.suptitle("ERA-5 t vs u — vmin/vmax comparison, 22 Apr 2026 12 UTC")
plt.tight_layout()
plt.show()



In [ ]:

from pandas.plotting import scatter_matrix
sel = dict(valid_time=ANALYSIS_TIME,latitude=slice(60, 35),    # high first — latitude descends
           longitude=slice(0, 30))

sub = ds['t'].sel(**sel)   # dims now: (pressure_level, lat, lon)

# build a pressure field matching the flattened grid
pres3d = sub.pressure_level.broadcast_like(sub).values.flatten()

data = {
    'pressure': pres3d,
    't':    ds['t'].sel(**sel).values.flatten() - 273.15,
    'q':    ds['q'].sel(**sel).values.flatten(),
    'u':    ds['u'].sel(**sel).values.flatten(),
    'v':    ds['v'].sel(**sel).values.flatten(),
    'w':    ds['w'].sel(**sel).values.flatten(),
    'ciwc': ds['ciwc'].sel(**sel).values.flatten(),
    'clwc': ds['clwc'].sel(**sel).values.flatten(),
}

df_wider = pd.DataFrame(data).dropna()

x = df_wider['t']
y = df_wider['u']
pressure = df_wider['pressure']

# ---- 1. Different COLOUR maps ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, cmap in zip(axes, ['viridis_r', 'coolwarm_r', 'plasma_r']):
    sc = ax.scatter(x, y, c=pressure, cmap=cmap, s=15, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='pressure (hPa)')
    ax.set_title(f"cmap = '{cmap}'")
    ax.set_xlabel('t'); ax.set_ylabel('u')
plt.tight_layout(); plt.show()

# ---- 2. Different VALUE RANGES (vmin/vmax) ----
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ranges = [(0, 1000), (700, 1000), (850, 1000)]
for ax, (vmin, vmax) in zip(axes, ranges):
    sc = ax.scatter(x, y, c=pressure, cmap='viridis_r',
                    vmin=vmin, vmax=vmax, s=15, alpha=0.6)
    plt.colorbar(sc, ax=ax, label='pressure (hPa)')
    ax.set_title(f"range {vmin}-{vmax}")
    ax.set_xlabel('t'); ax.set_ylabel('u')
plt.tight_layout(); plt.show()



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# variables to plot
variables = ['pressure', 't', 'q', 'u', 'v', 'w']

# ---- define value-range filters (sub-task 1 & 2) ----
# each filter: condition (boolean mask), colour, marker
filters = [
    {
        'name': 'pressure 500-850 & t>0',
        'mask': (df_wider['pressure'].between(850, 1000)) & (df_wider['t'] > 0),
        'color': 'green',
        'marker': '+',
    },
    {
        'name': 'eastward wind 20-35',
        'mask': df_wider['u'].between(20, 35),
        'color': 'red',
        'marker': 'x',
    },
]

# ---- build scatterplot matrix with highlighting ----
n = len(variables)
fig, axes = plt.subplots(n, n, figsize=(16, 16))

for i, yvar in enumerate(variables):
    for j, xvar in enumerate(variables):
        ax = axes[i, j]
        # base layer: all points, faint grey
        ax.scatter(df_wider[xvar], df_wider[yvar], s=3, color='lightgrey', alpha=0.3)
        # highlight layers: each filter on top
        for f in filters:
            m = f['mask']
            ax.scatter(df_wider[xvar][m], df_wider[yvar][m],
                       s=20, color=f['color'], marker=f['marker'], alpha=0.8)
        if i == n-1: ax.set_xlabel(xvar)
        if j == 0:   ax.set_ylabel(yvar)

# legend
for f in filters:
    axes[0, 0].scatter([], [], color=f['color'], marker=f['marker'], label=f['name'])
axes[0, 0].legend(fontsize=7, loc='upper right')

plt.suptitle('Brushing & Linking — highlighted value ranges')
plt.tight_layout()
plt.show()
print(df_wider.shape)

In [ ]:
import matplotlib.pyplot as plt

def build_df(ds, lat_slice, lon_slice):
    sel = dict(valid_time=ANALYSIS_TIME,
               latitude=lat_slice, longitude=lon_slice)
    sub = ds['t'].sel(**sel)
    pres = sub.pressure_level.broadcast_like(sub).values.flatten()
    data = {
        'pressure': pres,
        't':  ds['t'].sel(**sel).values.flatten() - 273.15,
        'q':  ds['q'].sel(**sel).values.flatten(),
        'u':  ds['u'].sel(**sel).values.flatten(),
        'v':  ds['v'].sel(**sel).values.flatten(),
        'w':  ds['w'].sel(**sel).values.flatten(),
    }
    return pd.DataFrame(data).dropna()

# (a) Germany   (b) North Atlantic   — lat descends, lon ascends
germany       = (slice(55, 47),  slice(6, 15))
north_atlantic = (slice(60, 40), slice(310, 350))   # ~50W–10W

df_de = build_df(ds, *germany)
df_na = build_df(ds, *north_atlantic)

In [ ]:
import matplotlib.pyplot as plt

variables = ['pressure', 't', 'q', 'u', 'v', 'w']

filters = [
    {'name': 'low level (850-1000 hPa)',  'mask_col': 'pressure', 'lo': 850, 'hi': 1000,
     'color': 'green',  'marker': '+'},
    {'name': 'high level (200-300 hPa)', 'mask_col': 'pressure', 'lo': 200, 'hi': 300,
     'color': 'red',    'marker': 'x'},
]

def brush_matrix(df, title):
    n = len(variables)
    fig, axes = plt.subplots(n, n, figsize=(15, 15))
    for i, yv in enumerate(variables):
        for j, xv in enumerate(variables):
            ax = axes[i, j]
            ax.scatter(df[xv], df[yv], s=3, color='lightgrey', alpha=0.3)
            for f in filters:
                m = df[f['mask_col']].between(f['lo'], f['hi'])
                ax.scatter(df[xv][m], df[yv][m], s=15,
                           color=f['color'], marker=f['marker'], alpha=0.7)
            if i == n-1: ax.set_xlabel(xv)
            if j == 0:   ax.set_ylabel(yv)
    for f in filters:
        axes[0,0].scatter([], [], color=f['color'], marker=f['marker'], label=f['name'])
    axes[0,0].legend(fontsize=6)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

brush_matrix(df_de, "Germany — low vs high pressure layers")
brush_matrix(df_na, "North Atlantic — low vs high pressure layers")

In [ ]:
import matplotlib.pyplot as plt

variables = ['pressure', 't', 'q', 'u', 'v', 'w']

filters = [
    {'name': 'weak eastward wind (0-20 m/s)', 'mask_col': 'u', 'lo': 0,  'hi': 20,
     'color': 'blue',   'marker': '+'},
    {'name': 'strong eastward wind (>50 m/s)','mask_col': 'u', 'lo': 50, 'hi': 999,
     'color': 'orange', 'marker': 'x'},
]

def brush_matrix(df, title):
    n = len(variables)
    fig, axes = plt.subplots(n, n, figsize=(15, 15))
    for i, yv in enumerate(variables):
        for j, xv in enumerate(variables):
            ax = axes[i, j]
            ax.scatter(df[xv], df[yv], s=3, color='lightgrey', alpha=0.3)
            for f in filters:
                m = df[f['mask_col']].between(f['lo'], f['hi'])
                ax.scatter(df[xv][m], df[yv][m], s=15,
                           color=f['color'], marker=f['marker'], alpha=0.7)
            if i == n-1: ax.set_xlabel(xv)
            if j == 0:   ax.set_ylabel(yv)
    for f in filters:
        axes[0,0].scatter([], [], color=f['color'], marker=f['marker'], label=f['name'])
    axes[0,0].legend(fontsize=6)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

brush_matrix(df_de, "Germany — weak vs strong eastward wind")
brush_matrix(df_na, "North Atlantic — weak vs strong eastward wind")

In [ ]:
print(ds['u'].attrs)
print(ds['v'].attrs)

## Conclusion

This notebook demonstrates a practical workflow for analysing ERA5 weather data with Python. It shows how to inspect NetCDF metadata, select a geographic grid point, visualise global and regional temperature fields, compare interpolation methods, and explore relationships between multiple atmospheric variables.

